Este notebook corresponde a la primera parte del notebook 3, donde se realiza agregación temporal, sumando los embeddings por TR y la construcción de representaciones FIR.

In [41]:
import numpy as np
import os
from scipy.spatial.distance import cosine
import scipy.io as sio

In [42]:
#Verficación de los archivos y sus dimensiones
embed_files = [f for f in os.listdir('.') if f.startswith('embeddings_') and f.endswith('_layer16.npy')]
for f in sorted(embed_files):
    e = np.load(f)
    print(f, e.shape)

embeddings_cognitive_layer16.npy (5176, 4096)
embeddings_emotional_layer16.npy (5176, 4096)
embeddings_literary_layer16.npy (5176, 4096)
embeddings_neutral_layer16.npy (5176, 4096)
embeddings_spatial_layer16.npy (5176, 4096)
embeddings_syntactic_layer16.npy (5176, 4096)


**Verficación de la distancia entre condiciones**

El objetivo de esta celda es una verificación ("sanity check") antes de seguir con el procesamiento de que los embeddings generados por prompt son distintos. 

In [54]:
word_texts = list(np.load('word_texts.npy', allow_pickle=True))
print(f'Archivos de embeddings encontrados: {len(embed_files)}\n')

test_word_idx = min(1725, len(word_texts) - 1)
print(f'Palabra de test (índice {test_word_idx}): "{word_texts[test_word_idx]}"\n')

test_embeds = {}

for f in sorted(embed_files):
    name = f.replace('embeddings_', '').replace('_layer16.npy', '')
    e = np.load(f)

    test_embeds[name] = e[test_word_idx]

    print(f'{name:15s}: shape={e.shape}, '
          f'norma media={np.linalg.norm(e, axis=1).mean():.2f}')

print('\nDistancia coseno entre condiciones (palabra test):')

names = sorted(test_embeds.keys())

for i, n1 in enumerate(names):
    for n2 in names[i+1:]:
        dist = cosine(test_embeds[n1], test_embeds[n2])
        print(f'  {n1} vs {n2}: {dist:.4f}')

Archivos de embeddings encontrados: 6

Palabra de test (índice 1725): ""HARRY"

cognitive      : shape=(5176, 4096), norma media=10.22
emotional      : shape=(5176, 4096), norma media=10.26
literary       : shape=(5176, 4096), norma media=10.27
neutral        : shape=(5176, 4096), norma media=10.23
spatial        : shape=(5176, 4096), norma media=10.20
syntactic      : shape=(5176, 4096), norma media=10.26

Distancia coseno entre condiciones (palabra test):
  cognitive vs emotional: 0.0043
  cognitive vs literary: 0.0096
  cognitive vs neutral: 0.0169
  cognitive vs spatial: 0.0036
  cognitive vs syntactic: 0.0157
  emotional vs literary: 0.0178
  emotional vs neutral: 0.0103
  emotional vs spatial: 0.0037
  emotional vs syntactic: 0.0266
  literary vs neutral: 0.0374
  literary vs spatial: 0.0114
  literary vs syntactic: 0.0056
  neutral vs spatial: 0.0129
  neutral vs syntactic: 0.0427
  spatial vs syntactic: 0.0166


Se confirma que las diferentes condiciones generan archivos distintos, con distancias de coseno que oscilan entre 0.14 y 0.75 para una misma palabra. Se observa una clara diferencia entre el prompt "neutral" y todo el resto; la condición neutral genera un espacio representacional significativamente distinto

**Suma de embeddings por TR**

La siguiente celda agrega los embeddings de las ~4 palabras de cada TR, y devuelve una matriz (nTRs, embed_dim) con la misma escala temporal que los datos fMRI: un vector por cada 2 segundos.

In [46]:
mat = sio.loadmat(f'subject_1.mat', squeeze_me=True)
word_to_tr = np.load('word_to_tr.npy')
time_info = mat['time']
time_fmri = time_info[:, 0]
n_trs = len(time_fmri)

In [47]:
PROMPTS = {
    'neutral': "",  # Sin instrucción, baseline
    'literary': (
        "You are a literary critic. As you read this text, focus on "
        "narrative structure, character development, and prose style."
    ),
    'cognitive': (
        "You are a cognitive scientist. As you read, focus on the mental "
        "states, beliefs, desires, and intentions of each character."
    ),
    'emotional': (
        "You are reading this story with deep emotional engagement. "
        "Focus on the emotions felt by each character and the emotional "
        "tension of each scene."
    ),
    'syntactic': (
        "You are a linguist analyzing sentence structure. Focus on "
        "grammatical complexity, dependency relations, and word order."
    ),
    'spatial': (
        'Pay close attention to physical movements, spatial locations, '
        'body positions, and the physical environment described in the text.'
    ),
}

In [48]:
def sum_embeddings_per_tr(embeddings, word_to_tr, n_trs):
    embed_dim = embeddings.shape[1]
    tr_embeds = np.zeros((n_trs, embed_dim))

    for w, tr in enumerate(word_to_tr):
        tr_embeds[tr] += embeddings[w]

    return tr_embeds

In [50]:
for prompt_name in PROMPTS:
    embeds = np.load(f"embeddings_{prompt_name}_layer16.npy")
    tr_embeds= sum_embeddings_per_tr(embeds, word_to_tr, n_trs)
    np.save(f"tr_embeddings_{prompt_name}.npy", tr_embeds)
    print(f"{prompt_name}: {tr_embeds.shape}")

neutral: (1351, 4096)
literary: (1351, 4096)
cognitive: (1351, 4096)
emotional: (1351, 4096)
syntactic: (1351, 4096)
spatial: (1351, 4096)


Las dimensiones de los archivos han descendido a la dimensión de los TRs, quedan alineados con los sujetos (matrices de dimensión (1351,10000))

**Máscara de TRs válidos**

No todos los TRs son válidos; es necesario excluir los TRs de fijación (correspondientes a aquellos en los que los sujetos estaban mirando una cruz en la pantalla, antes de comenzar la lectura), y los primeros 4 TRs con palabras de cada run, para la matriz FIR (explicada en la siguiente celda).

In [51]:
trs_with_words = set(word_to_tr)
fixation_mask = np.array([t not in trs_with_words for t in range(len(time_fmri))])
runs = time_info[:, 1].astype(int)

valid_trs = np.ones(n_trs, dtype=bool)

#TRs de fijación
valid_trs[fixation_mask] = False

#Los 4 primeros TRs con palabras de cada run
for run_id in np.unique(runs):
    run_word_trs = np.where ((runs == run_id) & ~fixation_mask)[0]
    if len(run_word_trs) >= 4:
        valid_trs[run_word_trs[:4]] = False

np.save('valid_trs.npy', valid_trs)
print(f"TRs válidos: {valid_trs.sum()} de {n_trs}")

TRs válidos: 1283 de 1351


**Construcción de la matriz FIR**

Es una matriz que se aplica al modelo para ajustarse a la manera de procesar del cerebro, debido al retraso hemodinámico. Es la razón por la que en la máscara de TRs válidos, que se aplicará tanto a sujetos como al modelo, se han descartado los 4 primeros TRs con palabras de cada run: se descartan palabras cuya relación con la respuesta cerebral no se puede modelar de manera limpia.

El modelo FIR (Finite Impulse Response) captura el retraso hemodinámico sin asumir una forma fija, supone una mejora metodológica frente al HRF tradicional (Hemodynamic Response Function, por sus siglas en inglés), que modela el retraso como una función doble gamma. 

En el caso del modelo FIR, el cerebro en el vóxel v, tiempo t, se modela como una combinación lineal de las representaciones lingüísticas de los momentos pasados, establecidos en los 4 TRs anteriores (puesto que el pico de la señal BOLD ocurre a los 4-6s, se establecen 4 delays (a 2, 4, 6, 8 segundos = 1, 2, 3, 4 TRs)) siguiendo la siguiente ecuación:
$$
y_v(t) = \sum_{j} \sum_{k} f_j(t-k)\; w^{k}_{vj}
$$

Es la utilizada por Wehbe et al. (2019), Appendix C de File S1. 

In [52]:
def create_fir_matrix(features, runs, n_delays=4):
   
    T, N = features.shape
    X_fir = np.zeros((T, N * n_delays))

    for d in range(1, n_delays + 1):
        for t in range(d, T):
            if runs[t] == runs[t - d]:
                X_fir[t, (d-1)*N : d*N] = features[t - d]
    return X_fir

In [53]:
for prompt_name in PROMPTS:
    tr_embeds = np.load(f'tr_embeddings_{prompt_name}.npy')
    X_fir = create_fir_matrix(tr_embeds, runs, n_delays=4)
    np.save(f'fir_matrix_{prompt_name}.npy', X_fir)
    print(f"{prompt_name}: FIR shape = {X_fir.shape}")

neutral: FIR shape = (1351, 16384)
literary: FIR shape = (1351, 16384)
cognitive: FIR shape = (1351, 16384)
emotional: FIR shape = (1351, 16384)
syntactic: FIR shape = (1351, 16384)
spatial: FIR shape = (1351, 16384)


Sanity check

In [1]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

SAVE_PATH = 'C:/Users/Ale/'   # pon aquí la carpeta de los .npy si no estás en ella, p.ej. 'C:/Users/Ale/'
PROMPTS = ['neutral','literary','cognitive','emotional','syntactic','spatial']

E   = {p: np.load(f'{SAVE_PATH}embeddings_{p}_layer16.npy') for p in PROMPTS}
neu = E['neutral']

print("=== 1) ¿Cuánto cambia el prompt el embedding de palabra (vs neutral)? ===")
for p in PROMPTS:
    if p == 'neutral': continue
    a, b = E[p], neu
    cos   = (a*b).sum(1) / (np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1))
    shift = np.linalg.norm(a-b, axis=1) / np.linalg.norm(b, axis=1)
    print(f"  {p:10s}: coseno medio con neutral={cos.mean():.4f}  |  desplazamiento |Δ|/|neutral|={shift.mean():.3f}")

print("\n=== 2) ¿Es distinta la GEOMETRÍA entre condiciones? (correlación entre RDMs) ===")
word_to_tr = np.load(f'{SAVE_PATH}word_to_tr.npy')
valid_trs  = np.load(f'{SAVE_PATH}valid_trs.npy')
def tr_sum(e):
    out = np.zeros((valid_trs.shape[0], e.shape[1]))
    for w, tr in enumerate(word_to_tr): out[tr] += e[w]
    return out
rdms = {p: squareform(pdist(tr_sum(E[p])[valid_trs], metric='correlation')) for p in PROMPTS}
vneu = squareform(rdms['neutral'])
for p in PROMPTS:
    if p == 'neutral': continue
    r, _ = spearmanr(squareform(rdms[p]), vneu)
    print(f"  RDM {p:10s} vs RDM neutral: Spearman={r:.4f}")

print("\n=== 3) ¿El efecto del prompt es un desplazamiento casi constante (invisible para RSA) o depende del contenido? ===")
for p in PROMPTS:
    if p == 'neutral': continue
    D = E[p] - neu
    d_mean = D.mean(0); d_mean /= np.linalg.norm(d_mean)
    cosd = (D @ d_mean) / (np.linalg.norm(D, axis=1) + 1e-9)
    print(f"  {p:10s}: alineación media de los Δ con su dirección común = {cosd.mean():.3f}")

=== 1) ¿Cuánto cambia el prompt el embedding de palabra (vs neutral)? ===
  literary  : coseno medio con neutral=0.9724  |  desplazamiento |Δ|/|neutral|=0.221
  cognitive : coseno medio con neutral=0.9850  |  desplazamiento |Δ|/|neutral|=0.158
  emotional : coseno medio con neutral=0.9890  |  desplazamiento |Δ|/|neutral|=0.135
  syntactic : coseno medio con neutral=0.9714  |  desplazamiento |Δ|/|neutral|=0.222
  spatial   : coseno medio con neutral=0.9869  |  desplazamiento |Δ|/|neutral|=0.146

=== 2) ¿Es distinta la GEOMETRÍA entre condiciones? (correlación entre RDMs) ===
  RDM literary   vs RDM neutral: Spearman=0.9816
  RDM cognitive  vs RDM neutral: Spearman=0.9909
  RDM emotional  vs RDM neutral: Spearman=0.9935
  RDM syntactic  vs RDM neutral: Spearman=0.9764
  RDM spatial    vs RDM neutral: Spearman=0.9912

=== 3) ¿El efecto del prompt es un desplazamiento casi constante (invisible para RSA) o depende del contenido? ===
  literary  : alineación media de los Δ con su dirección c

In [2]:
import numpy as np, scipy.io as sio
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

# Embeddings de palabra y CENTRADO entre condiciones (quita el contenido compartido)
E      = {p: np.load(f'{SAVE_PATH}embeddings_{p}_layer16.npy') for p in PROMPTS}
common = np.stack([E[p] for p in PROMPTS], 0).mean(0)       # contenido común por palabra
Ec     = {p: E[p] - common for p in PROMPTS}                # desviación específica de cada prompt

word_to_tr = np.load(f'{SAVE_PATH}word_to_tr.npy')
valid_trs  = np.load(f'{SAVE_PATH}valid_trs.npy')
runs       = sio.loadmat(f'{SAVE_PATH}subject_1.mat', squeeze_me=True)['time'][:,1].astype(int)
n_trs      = len(valid_trs)

def tr_sum(e):
    out = np.zeros((n_trs, e.shape[1]))
    for w, tr in enumerate(word_to_tr): out[tr] += e[w]
    return out
def fir(f, runs, nd=4):
    T, N = f.shape; X = np.zeros((T, N*nd))
    for d in range(1, nd+1):
        for t in range(d, T):
            if runs[t] == runs[t-d]: X[t,(d-1)*N:d*N] = f[t-d]
    return X

model_rdms = {p: squareform(pdist(fir(tr_sum(Ec[p]), runs)[valid_trs], metric='correlation'))
              for p in PROMPTS}

brain = [np.load(f'{SAVE_PATH}data_subject{s}_isc10k.npy')[valid_trs] for s in range(1,9)]
bvec  = squareform(np.mean([squareform(pdist(b, metric='correlation')) for b in brain], 0))

print("RSE con la señal específica de cada prompt (contenido compartido eliminado):")
for p in PROMPTS:
    print(f"  {p:10s}: {spearmanr(squareform(model_rdms[p]), bvec)[0]:.4f}")

RSE con la señal específica de cada prompt (contenido compartido eliminado):
  neutral   : 0.0218
  literary  : 0.0207
  cognitive : 0.0138
  emotional : 0.0181
  syntactic : 0.0166
  spatial   : 0.0154
